In [1]:
# 1. LES DONNÉES : les 8 jours observés + le jour à prédire

JOURS = {
    "J1": {"ciel": "Soleil", "humidite": 65, "vent": "Faible", "joue": "Oui"},
    "J2": {"ciel": "Soleil", "humidite": 90, "vent": "Faible", "joue": "Non"},
    "J3": {"ciel": "Pluie",  "humidite": 70, "vent": "Faible", "joue": "Oui"},
    "J4": {"ciel": "Soleil", "humidite": 85, "vent": "Faible", "joue": "Non"},
    "J5": {"ciel": "Pluie",  "humidite": 80, "vent": "Fort",   "joue": "Non"},
    "J6": {"ciel": "Soleil", "humidite": 70, "vent": "Fort",   "joue": "Non"},
    "J7": {"ciel": "Pluie",  "humidite": 60, "vent": "Faible", "joue": "Oui"},
    "J8": {"ciel": "Soleil", "humidite": 75, "vent": "Faible", "joue": "Oui"},
}
DEMAIN = {"ciel": "Soleil", "humidite": 72, "vent": "Fort"}

print("LES 8 JOURS")
for nom, j in JOURS.items():
    print(nom, j)


# 2. LES OUTILS DE L'ARBRE (comme pour le decision tree)
#    Gini = 1 − Σ pₖ²   (0 = groupe pur, 0,5 = moitié-moitié)
#    Gain = Gini(départ) − Σ (nⱼ / n) × Gini(groupe j)

def compter(etiquettes):
    """['Oui','Non','Oui'] -> {'Oui': 2, 'Non': 1}"""
    c = {}
    for e in etiquettes:
        c[e] = c.get(e, 0) + 1
    return c


def gini(etiquettes):
    n = len(etiquettes)
    if n == 0:
        return 0.0
    return 1 - sum((k / n) ** 2 for k in compter(etiquettes).values())


def classe_majoritaire(etiquettes):
    c = compter(etiquettes)
    return max(c, key=c.get)


def texte_compte(etiquettes):
    c = compter(etiquettes)
    return f"{c.get('Oui', 0)} O / {c.get('Non', 0)} N"


def questions_candidates(variable, jours):
    """Toutes les questions oui/non possibles sur une variable."""
    if variable == "ciel":
        return [("ciel", "=", "Soleil")]
    if variable == "vent":
        return [("vent", "=", "Fort")]
    # humidité (numérique) : seuils = milieux entre deux valeurs voisines triées
    valeurs = sorted(set(JOURS[j]["humidite"] for j in jours))
    return [("humidite", "<=", (a + b) / 2) for a, b in zip(valeurs, valeurs[1:])]

def repond_oui(question, jour):
    variable, op, valeur = question
    if op == "=":
        return jour[variable] == valeur
    return jour[variable] <= valeur


def texte(question):
    variable, op, valeur = question
    if op == "=":
        return f"{variable} = {valeur} ?"
    return f"{variable} <= {valeur:g} ?"


# 3. APPRENDRE UN ARBRE (une seule question : max_depth = 1)
#    On teste toutes les questions des variables autorisées
#    et on garde celle au plus grand gain.

def apprendre_arbre(echantillon, variables):
    # 1. On récupère les étiquettes (Oui/Non) de départ et on calcule son impureté (Gini)
    etiquettes_depart = []
    for j in echantillon:
        etiquettes_depart.append(JOURS[j]["joue"])

    g0 = gini(etiquettes_depart)
    n = len(echantillon)
    meilleur = None

    # 2. On teste chaque variable et chaque question candidate
    for variable in variables:
        for q in questions_candidates(variable, echantillon):

            # Séparation simple des jours selon la question
            oui = []
            non = []
            for j in echantillon:
                decision = JOURS[j]["joue"]
                if repond_oui(q, JOURS[j]):
                    oui.append(decision)
                else:
                    non.append(decision)

            # Si un des groupes est vide, cette question n'est pas utile
            if len(oui) == 0 or len(non) == 0:
                continue

            # Calcul de l'impureté moyenne après la question, puis du gain
            g_moy = (len(oui) / n) * gini(oui) + (len(non) / n) * gini(non)
            gain = g0 - g_moy

            # On garde en mémoire la question qui donne le meilleur gain
            if meilleur is None or gain > meilleur["gain"]:
                meilleur = {
                    "question": q,
                    "gain": gain,
                    "feuille_oui": classe_majoritaire(oui),
                    "feuille_non": classe_majoritaire(non)
                }

    return meilleur


def predire_arbre(arbre, jour):
    if repond_oui(arbre["question"], jour):
        return arbre["feuille_oui"]
    return arbre["feuille_non"]

# 4. LA FORÊT DE LA PRÉSENTATION : 3 arbres, 3 échantillons bootstrap


ECHANTILLONS = [
    ["J1", "J3", "J3", "J5", "J6", "J6", "J8", "J2"],   # arbre 1
    ["J2", "J2", "J4", "J5", "J7", "J7", "J8", "J1"],   # arbre 2
    ["J1", "J4", "J4", "J5", "J5", "J6", "J8", "J3"],   # arbre 3
]
VARIABLES = [
    ["ciel", "vent"],        # arbre 1 : m = 2 variables tirées
    ["ciel", "humidite"],    # arbre 2 : il n'a pas le droit de regarder le vent
    ["ciel", "vent"],        # arbre 3
]

foret = []
for i, (ech, variables) in enumerate(zip(ECHANTILLONS, VARIABLES), start=1):
    oob = sorted(set(JOURS) - set(ech))
    print(f"\n=== ARBRE {i} — échantillon {ech}")
    print(f"  variables autorisées : {variables} ; out-of-bag : {oob}")
    arbre = apprendre_arbre(ech, variables)
    if arbre:
        print(f"  -> question gardée : {texte(arbre['question'])} (gain {arbre['gain']:.3f}) ; "
              f"oui -> {arbre['feuille_oui']} ; non -> {arbre['feuille_non']}")
    foret.append(arbre)

# 5. PRÉDIRE DEMAIN : chaque arbre vote, la majorité l'emporte

def predire_foret(foret, jour, verbose=True):
    votes = [predire_arbre(a, jour) for a in foret]
    if verbose:
        for i, (a, v) in enumerate(zip(foret, votes), start=1):
            print(f"  Arbre {i} ({texte(a['question'])}) -> {v}")
        print(f"  Votes : {votes.count('Oui')} Oui / {votes.count('Non')} Non")
    return classe_majoritaire(votes)

print(f"\n=== PRÉDICTION pour demain : {DEMAIN} ===")
final = predire_foret(foret, DEMAIN)
print(f"  Réponse de la forêt : {final} -> {'on joue' if final == 'Oui' else 'on ne joue pas'}")


# 6. ERREUR OUT-OF-BAG (OOB)
#    Chaque jour est testé uniquement par les arbres qui ne l'ont pas vu.

def score_oob(foret, echantillons, verbose=True):
    bons, total = 0, 0
    for nom, jour in JOURS.items():
        arbres = [a for a, ech in zip(foret, echantillons) if nom not in ech]
        if not arbres:
            continue                         # jour vu par tous les arbres
        votes = [predire_arbre(a, jour) for a in arbres]
        pred = classe_majoritaire(votes)
        ok = pred == jour["joue"]
        bons += ok
        total += 1
        if verbose:
            print(f"  {nom} : testé par {len(arbres)} arbre(s), votes {votes} -> {pred} "
                  f"(vrai : {jour['joue']}) {'✓' if ok else '✗'}")
    return bons / total if total else 0.0

print("\n=== ERREUR OUT-OF-BAG ===")
print(f"  Exactitude OOB : {score_oob(foret, ECHANTILLONS):.0%}")


# 7. UNE VRAIE FORÊT : la forêt fait elle-même ses tirages au hasard
#    Petit générateur pseudo-aléatoire écrit à la main (sans random)

class Hasard:
    """Générateur congruentiel linéaire : x <- (a·x + c) mod m. Même graine -> mêmes tirages."""
    def __init__(self, graine=42):
        self.x = graine

    def suivant(self):                     # nombre dans [0, 1[
        self.x = (1103515245 * self.x + 12345) % 2 ** 31
        return self.x / 2 ** 31

    def entier(self, n):                   # entier dans 0 .. n-1
        return int(self.suivant() * n)

    def bootstrap(self, elements):         # n tirages AVEC remise
        return [elements[self.entier(len(elements))] for _ in elements]

    def sous_ensemble(self, elements, m):  # m éléments SANS remise
        reste = list(elements)
        return [reste.pop(self.entier(len(reste))) for _ in range(m)]


def entrainer_foret(n_arbres=25, max_features=2, graine=42):
    h = Hasard(graine)
    noms = list(JOURS)
    arbres, echantillons = [], []
    for _ in range(n_arbres):
        ech = h.bootstrap(noms)
        if len(set(JOURS[j]["joue"] for j in ech)) < 2:   # échantillon pur : rien à apprendre
            continue
        variables = h.sous_ensemble(["ciel", "humidite", "vent"], max_features)
        arbre = apprendre_arbre(ech, variables)
        if arbre is None:                                 # aucune question ne coupe
            continue
        arbres.append(arbre)
        echantillons.append(ech)
    return arbres, echantillons

grande_foret, ech_grande = entrainer_foret(n_arbres=25, max_features=2, graine=42)
print(f"\n=== VRAIE FORÊT : {len(grande_foret)} arbres appris ===")
for i, a in enumerate(grande_foret[:5], start=1):
    print(f"  arbre {i} : {texte(a['question'])} -> oui : {a['feuille_oui']} / non : {a['feuille_non']}")
print("  ...")
votes = [predire_arbre(a, DEMAIN) for a in grande_foret]
print(f"  Demain : {votes.count('Oui')} Oui / {votes.count('Non')} Non -> {classe_majoritaire(votes)}")
print(f"  Exactitude OOB : {score_oob(grande_foret, ech_grande, verbose=False):.0%}")

LES 8 JOURS
J1 {'ciel': 'Soleil', 'humidite': 65, 'vent': 'Faible', 'joue': 'Oui'}
J2 {'ciel': 'Soleil', 'humidite': 90, 'vent': 'Faible', 'joue': 'Non'}
J3 {'ciel': 'Pluie', 'humidite': 70, 'vent': 'Faible', 'joue': 'Oui'}
J4 {'ciel': 'Soleil', 'humidite': 85, 'vent': 'Faible', 'joue': 'Non'}
J5 {'ciel': 'Pluie', 'humidite': 80, 'vent': 'Fort', 'joue': 'Non'}
J6 {'ciel': 'Soleil', 'humidite': 70, 'vent': 'Fort', 'joue': 'Non'}
J7 {'ciel': 'Pluie', 'humidite': 60, 'vent': 'Faible', 'joue': 'Oui'}
J8 {'ciel': 'Soleil', 'humidite': 75, 'vent': 'Faible', 'joue': 'Oui'}

=== ARBRE 1 — échantillon ['J1', 'J3', 'J3', 'J5', 'J6', 'J6', 'J8', 'J2']
  variables autorisées : ['ciel', 'vent'] ; out-of-bag : ['J4', 'J7']
  -> question gardée : vent = Fort ? (gain 0.300) ; oui -> Non ; non -> Oui

=== ARBRE 2 — échantillon ['J2', 'J2', 'J4', 'J5', 'J7', 'J7', 'J8', 'J1']
  variables autorisées : ['ciel', 'humidite'] ; out-of-bag : ['J3', 'J6']
  -> question gardée : humidite <= 77.5 ? (gain 0.500) 